# 3d print Product Creation - Setup Phase

This notebook sets up all the necessary components for creating the 3d printed product with DPP.

**Flow mirrors the GUI CreateProjectForm.tsx:**
1. User authentication and registration
2. Location setup
3. Units and resource specifications
4. Process definitions
5. Initial resources and components creation
6. Image asset preparation

In [1]:
import os
os.environ['IF_KEY'] = 'use you own key'


In [2]:
SECRET_KEY = os.environ.get('IF_KEY')

In [3]:
SECRET_KEY

'use you own key'

In [4]:
# Module imports and auto-reload setup
%load_ext autoreload
%aimport if_lib, if_utils, if_dpp, if_graphics, if_consts, if_gc1dpp
%autoreload 1
import os
import json
import random
import csv
from pathlib import Path

from if_utils import get_filename, show_data, save_traces

from if_lib import generate_random_challenge, read_HMAC, read_keypair, get_id_person, get_location_id, \
get_unit_id, get_resource_spec_id, get_resource, get_process, create_event, make_transfer, reduce_resource, set_user_location

from if_gc1dpp import upload_file_on_dpp, calculate_file_checksum

## Influx Fetching for dynamic value integration

In [ ]:
from influxdb_client import InfluxDBClient
import pandas as pd

# --- Connection details ---
url = "http://openlab.fchh.lauds.zenr.io:8086"

token = "update your own token"   # !!!clean before checkin!!!
org = "microfactory"
bucket = "openlab-microfactory"
machine_id = "prusa-mk4-1"
# --- Connect to InfluxDB ---
client = InfluxDBClient(url=url, token=token, org=org)

query = f'''
from(bucket: "{bucket}")
  |> range(start: 0)
  |> filter(fn: (r) => r["_measurement"] == "jobs")
  |> filter(fn: (r) => r["machineID"] == "{machine_id}")
  |> filter(fn: (r) => r["_field"] == "energyConsumed" or r["_field"] == "filament used [g]" or r["_field"] == "job_name" or r["_field"] == "jobDuration" or r["_field"] == "jobId" or r["_field"] == "filament_type")
  |> pivot(rowKey: ["_time"],columnKey: ["_field"],valueColumn: "_value")
'''
query_api = client.query_api()
tables = query_api.query_data_frame(org=org, query=query)
if isinstance(tables, list):
    df = pd.concat(tables)
else:
    df = tables

# --- Keep only rows where ALL required fields are present (avoids mixing two different jobs) ---
required_cols = ["energyConsumed", "filament used [g]", "job_name",
                  "jobDuration", "jobId", "filament_type"]
existing_required = [c for c in required_cols if c in df.columns]

df_complete = df.dropna(subset=existing_required)
df_complete = df_complete.sort_values("_time")

# --- Latest complete job ---
# --- Latest complete job with jobDuration > 2 min ---
df_complete = df_complete[df_complete["jobDuration"] > 120]  # jobDuration in seconds
latest_job = df_complete.iloc[[-1]]


In [6]:
latest_job

,result,table,_start,_stop,_time,_measurement,brand,machineID,sensorID,type,energyConsumed,filament used [g],filament_type,jobDuration,jobId,job_name
24,_result,0,1970-01-01 00:00:00+00:00,2026-08-28 08:17:39.909267+00:00,2026-06-28 23:30:16.470000+00:00,jobs,prusa,prusa-mk4-1,SPPS-05,3dp,386.139,111.14,PLA,12999.144,399,Fischbrotchen SonntagAnal Beads splitted 2 v2_...


## Configuration and Endpoints

In [7]:
# Define constants for this use case
USE_CASE = f'{org}_{machine_id}_{df["jobId"].iloc[0]}_' #Space_Machine_Job_1, bucket-name_machineID_jobID e.g. -> TMDC_BambooP2S_JobID

# Zenflows API endpoint
ENDPOINT = 'https://proxy.dpp-staging.dnstest.dyne.org/zenflows/api'

# DPP service endpoint
DPP_URL = 'https://proxy.dpp-staging.dnstest.dyne.org/interfacer-dpp'

# CSV file with product data
# CSV_FILE = Path('/Users/alcibiade/dyne/if/20260127 LOCI LAMP product informaiton DPP DE.csv')

# Assets directory
#ASSETS_DIR = Path('/Users/alcibiade/dyne/if/Interfacer-notebook/assets')

# Participants for LoCI LAMP production
USERS = ['LAUDS_A']


print(f"Configuration:")
print(f"  Use Case: {USE_CASE}")
print(f"  Zenflows: {ENDPOINT}")
print(f"  DPP URL: {DPP_URL}")

Configuration:
  Use Case: microfactory_prusa-mk4-1_369_
  Zenflows: https://proxy.dpp-staging.dnstest.dyne.org/zenflows/api
  DPP URL: https://proxy.dpp-staging.dnstest.dyne.org/interfacer-dpp


## File Path Configuration

In [8]:
# Calculate names of settings files
USERS_FILE = get_filename('cred_users.json', ENDPOINT, USE_CASE)
LOCS_FILE = get_filename('loc_users.json', ENDPOINT, USE_CASE)
UNITS_FILE = get_filename('units_data.json', ENDPOINT, USE_CASE)
SPECS_FILE = get_filename('res_spec_data.json', ENDPOINT, USE_CASE)
DPP_FILE = get_filename('dpp_data.json', ENDPOINT, USE_CASE)
RES_FILE = get_filename('initial_resources.json', ENDPOINT, USE_CASE)
PROCESS_FILE = get_filename('process_data.json', ENDPOINT, USE_CASE)
IMAGES_FILE = get_filename('images_data.json', ENDPOINT, USE_CASE)

print(f"Data will be saved to:")
print(f"  Users: {USERS_FILE}")
print(f"  Locations: {LOCS_FILE}")
print(f"  Units: {UNITS_FILE}")
print(f"  Resource Specs: {SPECS_FILE}")
print(f"  DPP Data: {DPP_FILE}")
print(f"  Initial Resources: {RES_FILE}")
print(f"  Processes: {PROCESS_FILE}")
print(f"  Images: {IMAGES_FILE}")

Data will be saved to:
  Users: use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/cred_users.json
  Locations: use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/loc_users.json
  Units: use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/units_data.json
  Resource Specs: use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/res_spec_data.json
  DPP Data: use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/dpp_data.json
  Initial Resources: use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/initial_resources.json
  Processes: use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/process_data.json
  Images: use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/images_data.jso

## Initialize Data Structures

In [9]:
# Create data structures
process_data = {}
res_data = {}
event_seq = []
dpp_data = {}
images_data = {}

# Initialize or load user data - LoCI LAMP specific users
if os.path.isfile(USERS_FILE):
    with open(USERS_FILE,'r') as f:
        users_data = json.loads(f.read())
    print("Credentials file available for users")
else:
    users_data = {}
    # Tchibo - the brand owner
    users_data['LAUDS_A'] = {
      "userChallenges": {
        "whereParentsMet": "Hamburg",
        "nameFirstPet": "Kaffee",
        "nameFirstTeacher": "Hans",
        "whereHomeTown": "Hamburg",
        "nameMotherMaid": "Schmidt"
      },
      "name": "LAUDS A",
      "username": "LAUDS_A_username",
      "email": "LAUDS_A@LAUDS.de",
      "note": "LAUDS"
    }
    with open(USERS_FILE,'w') as f:
        json.dump(users_data, f)
    print("Created new user credentials file")

# Initialize or load location data
if os.path.isfile(LOCS_FILE):
    with open(LOCS_FILE,'r') as f:
        locs_data = json.loads(f.read())
    print("Location file available")
else:
    locs_data = {}
    # TMDC COBALT location
    locs_data['LAUDS_A'] = {
        "name": "LAUDS TMDC COBALT", 
        "lat": 41.3543647,
        "long": 2.0967732,
        "addr": "Carrer del Cobalt, 57, 08940 Cornellà de Llobregat, Barcelona, Spain",
        "note": "LAUDS TMDC COBALT"
    }
    with open(LOCS_FILE,'w') as f:
        json.dump(locs_data, f)
    print("Created new location file")

# Initialize units and specs
if os.path.isfile(UNITS_FILE):
    with open(UNITS_FILE,'r') as f:
        units_data = json.loads(f.read())
    print(f"Unit file available")
else:
    units_data = {}

if os.path.isfile(SPECS_FILE):
    with open(SPECS_FILE,'r') as f:
        res_spec_data = json.loads(f.read())
    print(f"Resource Spec file available")
else:
    res_spec_data = {}

Created new user credentials file
Created new location file


## Authentication Setup: HMAC Generation

In [10]:
# Read HMAC or get it from the server
for user in USERS:
    read_HMAC(USERS_FILE, users_data, user, endpoint=ENDPOINT)

Payload
{'query': 'mutation ($firstRegistration: Boolean!, $userData: JSONObject!){\n  \n        keypairoomServer(firstRegistration: $firstRegistration, userData: $userData)\n      \n      }', 'variables': {'firstRegistration': True, 'userData': '{"email": "LAUDS_A@LAUDS.de"}'}}
Variables
{'firstRegistration': True, 'userData': '{"email": "LAUDS_A@LAUDS.de"}'}
Result
<Response [200]>
JSON
{
  "data": null,
  "errors": [
    {
      "message": "email exists",
      "path": [
        "keypairoomServer"
      ],
      "locations": [
        {
          "line": 3,
          "column": 9
        }
      ]
    }
  ]
}
Payload
{'query': 'mutation ($firstRegistration: Boolean!, $userData: JSONObject!){\n  \n        keypairoomServer(firstRegistration: $firstRegistration, userData: $userData)\n      \n      }', 'variables': {'firstRegistration': False, 'userData': '{"email": "LAUDS_A@LAUDS.de"}'}}
Variables
{'firstRegistration': False, 'userData': '{"email": "LAUDS_A@LAUDS.de"}'}
Result
<Response

## Cryptographic Key Generation

In [11]:
# Read the keypair for each user
for user in USERS:
    read_keypair(USERS_FILE, users_data, user)

result: ZenResult(output='{"ecdh_public_key":"BAchdmVw7bkd3n/NZJgOt5gcQSyONrVB2cwW/8SBqMhVHwbPSnYpodgTOKZJ3BCsAKdzGwAzsYb1pmgLrQeNDSI=","eddsa_public_key":"4FGna9DL9hw4anySFp1a5NMuM4wynD6xw5S1XKKCWnpb","ethereum_address":"0x3fa74dCa3962CB94a0C724bC02C965B4fAE8F972","keyring":{"ecdh":"8bBbT3uSjWrXkIiIoFnD+FiyTij8zIG7Z2MSTe0XwE0=","eddsa":"69G5AMWZPHkdEnDJFoRcVRxguSk3z4nJ7vpp2Wgyx4dw","ethereum":"54a076246ca83d5898532361df75a98f4c56df73dc431aaf19a257a5d38446ef","reflow":"JtUOp0nKI/ICAKdTD5y9b47BnZG0omjW0fGIJ1s0uQ4=","schnorr":"XRzbRtxREFmHDhX//CMv7GKZsxkZRFHhstBurjhcrZI="},"reflow_public_key":"AIbz9rw7q+7pnD4WvL+8GXLD/oCXM3pOPxKdaMRaiRr1x39aWhrieHjsILQg1lgAA14CM/o8Vl3Y0AlEFiUgKOB3Hsvrv+cOQQ3IeX4aNLBOiMZThcyCexrX7kqCpJ40FT8nx4Rj+4ObntlSoyXXNaKYPNrBIUetm/8P0s2oZJ8ECAYEcruYIvP+3b7g2zcgCdVXtBgwmhr8otByDTZuTAhikXQh/GmArfViETntAhbUyQC+bGhiNkZPLoWkINae","schnorr_public_key":"Aay3/9Ftx6aMWueqiIDFazbuEACC7/5IR+SHzLlpW5gw+/QJohAxWlmiYGSXpPpZ","seed":"stable weapon soap cruise future desk retire ho

## User Registration in Zenflows

In [12]:
# Read or get id of the person
for user in USERS:
    get_id_person(USERS_FILE, users_data, user, endpoint=ENDPOINT)

## Location Registration and Assignment

In [13]:
# Read or get the location id and set user locations
for user in USERS:
    get_location_id(LOCS_FILE, users_data[user], locs_data, user, endpoint=ENDPOINT)
    set_user_location(USERS_FILE, users_data, locs_data, user, endpoint=ENDPOINT)

## Unit of Measurement Registration

In [14]:
# Get the ids of all units
get_unit_id(UNITS_FILE, users_data['LAUDS_A'], units_data, 'piece', 'u_piece', 'om2:one', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['LAUDS_A'], units_data, 'mass', 'mg', 'om2:miligram', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['LAUDS_A'], units_data, 'time', 'h', 'om2:hour', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['LAUDS_A'], units_data, 'energy', 'kWh', 'om2:kilowattHour', endpoint=ENDPOINT)

In [15]:
#print(tables)
#df.head()
job_json_data = latest_job.to_dict(orient='records')
print(job_json_data)
print(job_json_data[0]["machineID"])


[{'result': '_result', 'table': 0, '_start': Timestamp('1970-01-01 00:00:00+0000', tz='UTC'), '_stop': Timestamp('2026-08-28 08:17:39.909267+0000', tz='UTC'), '_time': Timestamp('2026-06-28 23:30:16.470000+0000', tz='UTC'), '_measurement': 'jobs', 'brand': 'prusa', 'machineID': 'prusa-mk4-1', 'sensorID': 'SPPS-05', 'type': '3dp', 'energyConsumed': 386.1389999999956, 'filament used [g]': '111.14', 'filament_type': 'PLA', 'jobDuration': 12999.144, 'jobId': '399', 'job_name': 'Fischbrotchen SonntagAnal Beads splitted 2 v2_0.4n_0.2mm_PLA_MK4IS_4h0m.bgcode'}]
prusa-mk4-1


## Map InfluxDB data with REA process data

In [16]:
# get the below data from influx

jobdata ={
    "machineID": job_json_data[0]["machineID"],
    "jobID": job_json_data[0]["jobId"],
    "elEnergyConsumption": job_json_data[0]["energyConsumed"],
    "jobDuration": job_json_data[0]["jobDuration"],
    "materialConsumption": job_json_data[0]["filament used [g]"], #map to tracked material usage, example filament for 3d printers
    "partname": job_json_data[0]["job_name"],
    "machineType": job_json_data[0]["type"],
    "materialType": job_json_data[0]["filament_type"], #map to tracked material tyoe, example filament for 3d printers
}

print(jobdata)

{'machineID': 'prusa-mk4-1', 'jobID': '399', 'elEnergyConsumption': 386.1389999999956, 'jobDuration': 12999.144, 'materialConsumption': '111.14', 'partname': 'Fischbrotchen SonntagAnal Beads splitted 2 v2_0.4n_0.2mm_PLA_MK4IS_4h0m.bgcode', 'machineType': '3dp', 'materialType': 'PLA'}


## Process definition

In [17]:
# Create the processes

# Create the process that wraps creating the 3D object “Luffy”
process_name = f'Manufacture_part' #{jobdata["partname"]}_with_machine_{jobdata["machineID"]}'
user_data = users_data['LAUDS_A']
note = f"3D printing process to create object {jobdata['partname']} performed by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Save process data to file
with open(PROCESS_FILE, 'w') as f:
    json.dump(process_data, f, indent=2)
print(f"Process data saved to {PROCESS_FILE}")

Process data saved to use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/process_data.json


In [18]:
jobdata["machineType"]

'3dp'

## Resource Specification Registration

In [19]:
# Read all the resource specifications for 3D printing workflow

# Machine Resource Specifaction importing from csv LAUDS equipment list, check mapping

# 3D Printer
if jobdata["machineType"] == "3dp":
    machine = f"Machine-3D-Printer_{jobdata['machineID']}"
    machine_note = 'Specification for the 3D printer used by the operator'
    machine_classification = 'https://www.wikidata.org/wiki/Q3834994'  # Wikidata: 3D printer
    material = f"Filament_{jobdata['materialType']}"
    material_note = 'Specification for 3D printing filament material'
    material_classification = 'https://www.wikidata.org/wiki/Q25467586'  # Wikidata: plastic (or adjust to specific filament type)
elif jobdata["machineType"] == "laser":
    machine = f"Machine-Laser-Cutter_{jobdata['machineID']}"
    machine_note = 'Specification for the lasercutter used by the operator'
    machine_classification = 'https://www.wikidata.org/wiki/Q124816652'  # Wikidata: laser cutter
elif jobdata["machineType"] == "cnc":
    machine = f"Machine-CNC-Mill_{jobdata['machineID']}"
    machine_note = 'Specification for the cnc-machine used by the operator'
    machine_classification = 'https://www.wikidata.org/wiki/Q13231055'  # Wikidata: cnc-machine
else:
    machine = f"Machine-Unknown_{jobdata['machineID']}"

name = machine
note = machine_note
classification = machine_classification
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['LAUDS_A'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Filament
name = material
note = material_note
classification = material_classification
default_unit_id = units_data['mass']['id']
get_resource_spec_id(SPECS_FILE, users_data['LAUDS_A'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# Electrical Energy
name = 'Electrical-Energy'
note = 'Specification for electrical energy consumed'
classification = 'https://www.wikidata.org/wiki/Q206799'  # Wikidata: electricity
default_unit_id = units_data['energy']['id']
get_resource_spec_id(SPECS_FILE, users_data['LAUDS_A'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# 3D Object
name = f"3D-Object_{jobdata['partname']}"
note = 'Specification for the 3D printed object'
classification = 'https://www.wikidata.org/wiki/Q223557'  # Wikidata: manufactured object
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['LAUDS_A'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

# G-code file specification
name = f"{jobdata['partname']}_gcode-file"
note = 'Specification for the 3D printing G-code file'
classification = 'https://www.wikidata.org/wiki/Q1073076'  # Wikidata: Computer file
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['LAUDS_A'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)


## Initial Raw Materials Creation

In [20]:
# We create the resources that will not be saved to file as it is assumed they are recreated at each run

# Machine will be used
res_name = machine
amount = 1
get_resource(res_data, res_spec_data, res_name, users_data['LAUDS_A'], event_seq, amount, endpoint=ENDPOINT)

# Material for machining
res_name = material
amount = jobdata["materialConsumption"]  # in g (adjust based on your workflow)
get_resource(res_data, res_spec_data, res_name, users_data['LAUDS_A'], event_seq, amount, endpoint=ENDPOINT)

# Electrical energy for machining
res_name = 'Electrical-Energy'
amount = jobdata["elEnergyConsumption"]  # in Wh (adjust based on your workflow)
get_resource(res_data, res_spec_data, res_name, users_data['LAUDS_A'], event_seq, amount, endpoint=ENDPOINT)

# 3D-gcode file
res_name = f"{jobdata['partname']}_gcode-file"
amount = 1
get_resource(res_data, res_spec_data, res_name, users_data['LAUDS_A'], event_seq, amount, endpoint=ENDPOINT)

# Save initial resources to file
with open(RES_FILE, 'w') as f:
    json.dump(res_data, f, indent=2)
print(f"Initial resources saved to {RES_FILE}")

Initial resources saved to use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/initial_resources.json


## Component Production

Produce the semi-finished components from raw materials.

In [21]:
# add time duration fetched from influx and add to resource note to be visualized in sankey
# Step 1: Use machine
cur_pros = process_data[process_name]
action = 'use'
event_note=f"Using machine_{jobdata['machineID']}"
amount = jobdata["jobDuration"] / 60
cur_res = res_data[machine]
effort_spec = {}
effort_spec['unit_id'] = res_spec_data[machine]['defaultUnit']
effort_spec['spec_id'] = res_spec_data[machine]['id']
effort_spec['amount'] = amount

event_id, ts = create_event(
    users_data['LAUDS_A'], 
    action, 
    event_note,
    amount=amount, 
    process=cur_pros,
    res_spec_data=res_spec_data, 
    existing_res=cur_res,
    effort_spec=effort_spec, 
    endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
# event_seq.append({'ts': ts, 'process_id': cur_pros['id'], 'name': cur_pros['name']})



In [22]:
# Define event consume for material
cur_res = action = event_note = amount = cur_pros = None
action = 'consume'
event_note = f'consume material for {jobdata["partname"]}'
amount = jobdata["materialConsumption"]  # example: 50 grams, adjust as needed
cur_pros = process_data[process_name]

cur_res = res_data[material]  # pre-existing

event_id, ts = create_event(
    provider=users_data['LAUDS_A'],
    action='consume',                   # action on existing resource
    note=event_note,
    amount=amount,
    process=cur_pros,
    res_spec_data=res_spec_data,
    existing_res=cur_res,              # use existing_res, NOT new_res
    endpoint=ENDPOINT
)
event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})


In [23]:
# Define event consume for electrical energy
cur_res = action = event_note = amount = cur_pros = None
action = 'consume'
event_note = f'consume electrical energy for {jobdata["partname"]}'
amount =  jobdata["elEnergyConsumption"]  # example: 500 Wh, adjust as needed
cur_pros = process_data[process_name]

cur_res = res_data['Electrical-Energy']

event_id, ts = create_event(users_data['LAUDS_A'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)

event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})


In [24]:
# Define event cite for 3D gcode file
cur_res = action = event_note = amount = cur_pros = None
action = 'cite'
event_note = f"cite {jobdata['partname']}_gcode-file"
amount = 1
cur_pros = process_data[process_name]

cur_res = res_data[f"{jobdata['partname']}_gcode-file"]

event_id, ts = create_event(users_data['LAUDS_A'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)

event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})


In [25]:
# Define event produce for 3D object
cur_res = action = event_note = amount = cur_pros = None
action = 'produce'
event_note = f'produce {jobdata["partname"]}'
amount = 1
cur_pros = process_data[process_name]

res_data[f'{jobdata["partname"]}'] = {
    "res_ref_id": f'{jobdata["partname"]}-{random.randint(0, 10000)}',
    "name": f'{jobdata["partname"]}',
    "spec_id": res_spec_data[f"3D-Object_{jobdata['partname']}"]['id']
}

print(res_data[f'{jobdata["partname"]}'])
cur_res = res_data[f'{jobdata["partname"]}']

event_id, ts = create_event(users_data['LAUDS_A'], action, event_note, amount=amount, process=cur_pros,
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)

event_seq.append({'ts': ts, 'event_id': event_id, 'action': action, 'res_name': cur_res['name'], 'res': cur_res['id']})
event_seq.append({'ts': ts, 'process_id':cur_pros['id'], 'name' : cur_pros['name']})

{'res_ref_id': 'Fischbrotchen SonntagAnal Beads splitted 2 v2_0.4n_0.2mm_PLA_MK4IS_4h0m.bgcode-5138', 'name': 'Fischbrotchen SonntagAnal Beads splitted 2 v2_0.4n_0.2mm_PLA_MK4IS_4h0m.bgcode', 'spec_id': '06G4EWJFQSCE9A91YF56ZX318R'}


## Save All Data

In [26]:
# ============================================================================
# SAVE ALL DATA FOR PRODUCTION NOTEBOOK
# ============================================================================

# Save all resources to file
with open(RES_FILE, 'w') as f:
    json.dump(res_data, f, indent=2)
print(f"✓ Resources saved to {RES_FILE}")

# Save process data
with open(PROCESS_FILE, 'w') as f:
    json.dump(process_data, f, indent=2)
print(f"✓ Process data saved to {PROCESS_FILE}")

# Save locations data (needed by Production for LocationStep)
LOCATIONS_FILE = get_filename('locations_data.json', ENDPOINT, USE_CASE)
with open(LOCATIONS_FILE, 'w') as f:
    json.dump(locs_data, f, indent=2)
print(f"✓ Locations saved to {LOCATIONS_FILE}")

# Save specs data (includes specDpp and locilamp_assembled)
with open(SPECS_FILE, 'w') as f:
    json.dump(res_spec_data, f, indent=2)
print(f"✓ Specs saved to {SPECS_FILE}")

# Save units data
with open(UNITS_FILE, 'w') as f:
    json.dump(units_data, f, indent=2)
print(f"✓ Units saved to {UNITS_FILE}")


print(f"\n📁 All data files saved.")

✓ Resources saved to use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/initial_resources.json
✓ Process data saved to use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/process_data.json
✓ Locations saved to use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/locations_data.json
✓ Specs saved to use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/res_spec_data.json
✓ Units saved to use_cases/microfactory_prusa-mk4-1_369_/proxy.dpp-staging.dnstest.dyne.org%2Fzenflows%2Fapi/units_data.json

📁 All data files saved.


## Setup Complete - Summary

In [27]:
print("\n" + "="*60)
print(f'produce {jobdata["partname"]} SETUP COMPLETE')
print("="*60)
print(f"\n✓ {len(users_data)} users registered")
print(f"✓ {len(locs_data)} locations created")
print(f"✓ {len(units_data)} units registered")
print(f"✓ {len(res_spec_data)} resource specifications created")
print(f"✓ {len(process_data)} processes defined")
print(f"✓ {len(res_data)} resources created")

print(f"\n✅ You can now run the Visualization.")


produce Fischbrotchen SonntagAnal Beads splitted 2 v2_0.4n_0.2mm_PLA_MK4IS_4h0m.bgcode SETUP COMPLETE

✓ 1 users registered
✓ 1 locations created
✓ 4 units registered
✓ 5 resource specifications created
✓ 1 processes defined
✓ 5 resources created

✅ You can now run the Visualization.


# visualization

In [28]:
# Calculate file paths
USERS_FILE = get_filename('cred_users.json', ENDPOINT, USE_CASE)
LOCS_FILE = get_filename('loc_users.json', ENDPOINT, USE_CASE)
UNITS_FILE = get_filename('units_data.json', ENDPOINT, USE_CASE)
SPECS_FILE = get_filename('res_spec_data.json', ENDPOINT, USE_CASE)
DPP_FILE = get_filename('dpp_data.json', ENDPOINT, USE_CASE)
RES_FILE = get_filename('initial_resources.json', ENDPOINT, USE_CASE)
PROCESS_FILE = get_filename('process_data.json', ENDPOINT, USE_CASE)

# Load all data
print("Loading setup data...")

with open(USERS_FILE, 'r') as f:
    users_data = json.loads(f.read())
print(f"✓ Loaded {len(users_data)} users")

with open(LOCS_FILE, 'r') as f:
    locs_data = json.loads(f.read())
print(f"✓ Loaded {len(locs_data)} locations")

with open(UNITS_FILE, 'r') as f:
    units_data = json.loads(f.read())
print(f"✓ Loaded {len(units_data)} units")

with open(SPECS_FILE, 'r') as f:
    res_spec_data = json.loads(f.read())
print(f"✓ Loaded {len(res_spec_data)} resource specifications")

with open(PROCESS_FILE, 'r') as f:
    process_data = json.loads(f.read())
print(f"✓ Loaded {len(process_data)} processes")

with open(RES_FILE, 'r') as f:
    res_data = json.loads(f.read())
print(f"✓ Loaded {len(res_data)} initial resources")

# Initialize event sequence and DPP data
event_seq = []
dpp_data = {}

print("\n✓ All setup data loaded successfully!")

Loading setup data...
✓ Loaded 1 users
✓ Loaded 1 locations
✓ Loaded 4 units
✓ Loaded 5 resource specifications
✓ Loaded 1 processes
✓ Loaded 5 initial resources

✓ All setup data loaded successfully!


In [29]:
res_data

{'Machine-3D-Printer_prusa-mk4-1': {'res_ref_id': 'Machine-3D-Printer_prusa-mk4-1-3959',
  'name': 'Machine-3D-Printer_prusa-mk4-1',
  'spec_id': '06G4EWJ98CQ15VA2W8VPMVXWQR',
  'id': '06G4EWJJDBXAN8HX5XQWWB1FNW'},
 'Filament_PLA': {'res_ref_id': 'Filament_PLA-4095',
  'name': 'Filament_PLA',
  'spec_id': '06G4EWJC4GAR3HJ87WDH4STVSM',
  'id': '06G4EWJN7NYS0V3DSS0M4PKHXW'},
 'Electrical-Energy': {'res_ref_id': 'Electrical-Energy-5412',
  'name': 'Electrical-Energy',
  'spec_id': '06G4EWJDWAMCKV3G7QRANQ2Q78',
  'id': '06G4EWJS7DDKSRRX06XQBEG5BC'},
 'Fischbrotchen SonntagAnal Beads splitted 2 v2_0.4n_0.2mm_PLA_MK4IS_4h0m.bgcode_gcode-file': {'res_ref_id': 'Fischbrotchen SonntagAnal Beads splitted 2 v2_0.4n_0.2mm_PLA_MK4IS_4h0m.bgcode_gcode-file-7143',
  'name': 'Fischbrotchen SonntagAnal Beads splitted 2 v2_0.4n_0.2mm_PLA_MK4IS_4h0m.bgcode_gcode-file',
  'spec_id': '06G4EWJGK5480P41FEDS1533AG',
  'id': '06G4EWJW7C0C36ZHNWX357VX4G'},
 'Fischbrotchen SonntagAnal Beads splitted 2 v2_0.4n_0.2

In [30]:
# Verify that all components are available
print("Checking for required components...")
required_components = [f'{jobdata["partname"]}']

all_present = True
for component in required_components:
    if component in res_data:
        print(f"✓ {component}: {res_data[component]['id']}")
    else:
        print(f"✗ {component}: NOT FOUND")
        all_present = False

if all_present:
    print(f"\n✓ All components loaded successfully! ")
else:
    print(f"\n✗ Some components are missing. Please run the setup notebook first.")

Checking for required components...
✓ Fischbrotchen SonntagAnal Beads splitted 2 v2_0.4n_0.2mm_PLA_MK4IS_4h0m.bgcode: 06G4EWK91AQQPG40154H3KE5JW

✓ All components loaded successfully! 


In [31]:
# Display summary of created resources and DPP
show_data(users_data, locs_data, res_data, units_data, res_spec_data, process_data, event_seq)

Users
{
  "LAUDS_A": {
    "userChallenges": {
      "whereParentsMet": "Hamburg",
      "nameFirstPet": "Kaffee",
      "nameFirstTeacher": "Hans",
      "whereHomeTown": "Hamburg",
      "nameMotherMaid": "Schmidt"
    },
    "name": "LAUDS A",
    "username": "LAUDS_A_username",
    "email": "LAUDS_A@LAUDS.de",
    "note": "LAUDS",
    "seedServerSideShard.HMAC": "myFogga3kMk94XZ4Z6AUJPKwEmmirOPZD9HTfg/PMJo=",
    "seed": "stable weapon soap cruise future desk retire hockey session stand swing merit",
    "eddsa_public_key": "4FGna9DL9hw4anySFp1a5NMuM4wynD6xw5S1XKKCWnpb",
    "keyring": {
      "eddsa": "69G5AMWZPHkdEnDJFoRcVRxguSk3z4nJ7vpp2Wgyx4dw"
    },
    "id": "06EJ7SHEC7RVNMSAKDYMJARP6M",
    "location_id": "06G4EWHPYXQ9YJTS1T252WX6Q0"
  }
}
Locations
{
  "LAUDS_A": {
    "name": "LAUDS TMDC COBALT",
    "lat": 41.3543647,
    "long": 2.0967732,
    "addr": "Carrer del Cobalt, 57, 08940 Cornell\u00e0 de Llobregat, Barcelona, Spain",
    "note": "LAUDS TMDC COBALT",
    "id": 

In [32]:
from if_dpp import trace_query, check_traces, er_before, get_dpp
from if_graphics import vis_dpp, make_sankey, consol_trace

In [33]:
# Trace the 3D object 
trace_me = res_data[f'{jobdata["partname"]}']['id']
print(f"Resource to be traced: {trace_me}")

tot_dpp = []
visited = set()

# Trace all related events and processes for this resource
er_before(trace_me, users_data['LAUDS_A'], dpp_children=tot_dpp, depth=0, visited=visited, endpoint=ENDPOINT)

# Serializing to JSON
json_object = json.dumps(tot_dpp, indent=2)

print(json_object)
print(visited)


Resource to be traced: 06G4EWK91AQQPG40154H3KE5JW
[
  {
    "accountingQuantity": {
      "hasNumericalValue": "1",
      "hasUnit": {
        "id": "06G4EWHTX20XRVP3ERY8N2N4G4",
        "label": "u_piece",
        "symbol": "om2:one"
      }
    },
    "currentLocation": {
      "alt": "0",
      "id": "06G4EWHPYXQ9YJTS1T252WX6Q0",
      "lat": "41.3543647",
      "long": "2.0967732",
      "mappableAddress": "Carrer del Cobalt, 57, 08940 Cornell\u00e0 de Llobregat, Barcelona, Spain",
      "name": "LAUDS TMDC COBALT",
      "note": "LAUDS TMDC COBALT"
    },
    "custodian": {
      "id": "06EJ7SHEC7RVNMSAKDYMJARP6M",
      "name": "LAUDS A",
      "note": null,
      "primaryLocation": {
        "alt": "0",
        "id": "06G4EWHPYXQ9YJTS1T252WX6Q0",
        "lat": "41.3543647",
        "long": "2.0967732",
        "mappableAddress": "Carrer del Cobalt, 57, 08940 Cornell\u00e0 de Llobregat, Barcelona, Spain",
        "name": "LAUDS TMDC COBALT",
        "note": "LAUDS TMDC COBALT"
 

In [34]:
be_dpp = get_dpp(trace_me, endpoint=ENDPOINT)
print(json.dumps(be_dpp, indent=2))

[
  {
    "node": {
      "accountingQuantity": {
        "hasNumericalValue": "1",
        "hasUnit": {
          "id": "06G4EWHTX20XRVP3ERY8N2N4G4"
        }
      },
      "classifiedAs": null,
      "conformsTo": {
        "id": "06G4EWJFQSCE9A91YF56ZX318R"
      },
      "containedIn": {
        "id": null
      },
      "currentLocation": {
        "id": "06G4EWHPYXQ9YJTS1T252WX6Q0"
      },
      "custodian": {
        "id": "06EJ7SHEC7RVNMSAKDYMJARP6M"
      },
      "id": "06G4EWK91AQQPG40154H3KE5JW",
      "license": null,
      "licensor": null,
      "lot": {
        "id": null
      },
      "metadata": null,
      "name": "Fischbrotchen SonntagAnal Beads splitted 2 v2_0.4n_0.2mm_PLA_MK4IS_4h0m.bgcode",
      "note": null,
      "okhv": null,
      "onhandQuantityHas": {
        "hasUnit": {
          "id": "06G4EWHTX20XRVP3ERY8N2N4G4"
        },
        "numericalValue": "1"
      },
      "previousEvent": {
        "id": "06G4EWK916HT8N3G57YAVQ9RW0"
      },
      "prima

In [35]:
trace = trace_query(trace_me, endpoint=ENDPOINT)
# check consistency between the registered events, the back-end trace and the generated dpp
check_traces(trace, event_seq, tot_dpp, be_dpp)

################################################################################
nr trace: 15, nr events: 0, nr front-end dpp: 15, nr back-end dpp: 15
################################################################################
Check whether there are any duplicated trace items
################################################################################
Check whether there are any duplicated events
################################################################################
Check whether there are any duplicated nodes in front-end dpp
################################################################################
Check whether there are any duplicated nodes in back-end dpp
################################################################################
Are trace items in the events?
NOT FOUND: trace item Fischbrotchen SonntagAnal Beads splitted 2 v2_0.4n_0.2mm_PLA_MK4IS_4h0m.bgcode id: 06G4EWK91AQQPG40154H3KE5JW of type EconomicResource
NOT FOUND: trace item produce id: 06

In [36]:
save_traces(USE_CASE, tot_dpp, trace, be_dpp, event_seq)

In [37]:
locid_to_addr = {loc['id']: loc['addr'] for loc in locs_data.values()}

def user_label(user_key, user_record):
    name = user_record.get('name', user_key)
    loc_id = user_record.get('location_id')
    address = locid_to_addr.get(loc_id, 'Unknown location')
    return f"{name} ({address})"

user_labels = {
    uid: user_label(uid, udata)
    for uid, udata in users_data.items()
}

In [38]:
labels = []
sources = []
targets = []
values = []
color_nodes = []
color_links = []
assigned = {}
vis_dpp(tot_dpp[0], count=0, assigned=assigned, labels=labels, targets=targets, sources=sources, values=values, color_nodes=color_nodes, color_links=color_links)
sources, targets = consol_trace(assigned, sources, targets)
print("Users:")
for uid, lbl in user_labels.items():
    print(f"  {lbl}")

make_sankey(sources, targets, labels, values, color_nodes, color_links)
# make_sankey([0,0,1,2,2], [2,3,3,3,4], ['0','1','2','3','4'], [2,1,1,1,1], color_nodes, color_links)

Users:
  LAUDS A (Carrer del Cobalt, 57, 08940 Cornellà de Llobregat, Barcelona, Spain)
